# Composed Image Retrieval: "Find This Dress But in Red"

**Goal:** Fine-tune a model for composed retrieval — where the query is a (reference image + modification text) pair, and the target is a matching image. We'll also build a **custom fusion model** that replaces BLIP-2's default embedding addition with a learned FFN.

**Setup:**
- **Base model:** `Salesforce/blip2-itm-vit-g` — BLIP-2 with joint image-text encoding
- **Dataset:** FashionIQ (dress category) — "here's a dress, find me one that's more red / shorter / floral"
- **Custom model:** Same BLIP-2 backbone, but with an FFN that learns to fuse image + text embeddings instead of just adding them

**Plan:**
1. Download FashionIQ data and build a `ComposedRetrievalDataset`
2. Evaluate baseline BLIP-2 (no fine-tuning)
3. Fine-tune BLIP-2 with LoRA using khoji's standard pipeline
4. Build a custom FFN fusion model on top of BLIP-2 and fine-tune it
5. Compare all three: baseline vs LoRA vs custom FFN

In [ ]:
!pip install khoji

## Step 1: Download FashionIQ Data and Build Dataset

FashionIQ provides (candidate_image, captions, target_image) triplets for fashion items. We'll convert these into khoji's `ComposedRetrievalDataset` format where each query is an (image, text) pair.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/fashioniq/download_data.py", "./data/fashioniq"], check=True)
print("FashionIQ data downloaded.")

In [ ]:
import json
from pathlib import Path
import khoji

CATEGORY = "dress"
DATA_DIR = "./data/fashioniq"
CACHE_DIR = "./data/fashioniq/image_cache"
MODEL = "Salesforce/blip2-itm-vit-g"
MAX_EVAL_QUERIES = 100

URL_MAP_BASE = (
    "https://raw.githubusercontent.com/"
    "hongwang600/fashion-iq-metadata/master/image_url"
)

# Load URL mapping (ASIN -> image URL)
def load_url_mapping(data_dir, category):
    cache_file = Path(data_dir) / "image_url" / f"asin2url.{category}.txt"
    if not cache_file.exists():
        from urllib.request import urlretrieve
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{URL_MAP_BASE}/asin2url.{category}.txt", cache_file)
    mapping = {}
    with open(cache_file) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                mapping[parts[0].strip()] = parts[1].strip()
    return mapping

# Convert FashionIQ to ComposedRetrievalDataset
def build_dataset(data_dir, category, split, url_mapping):
    data_path = Path(data_dir)
    with open(data_path / "captions" / f"cap.{category}.{split}.json") as f:
        annotations = json.load(f)
    with open(data_path / "image_splits" / f"split.{category}.{split}.json") as f:
        gallery_ids = json.load(f)

    queries, corpus, qrels = {}, {}, {}
    for gid in gallery_ids:
        if gid in url_mapping:
            corpus[gid] = (url_mapping[gid], "")  # image-only corpus items
    for i, ann in enumerate(annotations):
        if ann["candidate"] not in url_mapping or ann["target"] not in corpus:
            continue
        for cap_idx, caption in enumerate(ann["captions"]):
            qid = f"q_{i}_{cap_idx}"
            queries[qid] = (url_mapping[ann["candidate"]], caption)
            qrels[qid] = {ann["target"]: 1}

    ds = khoji.ComposedRetrievalDataset(queries=queries, corpus=corpus, qrels=qrels, base_dir=CACHE_DIR)
    print(f"{category}/{split}: {len(queries)} queries, {len(corpus)} gallery images")
    return ds

url_mapping = load_url_mapping(DATA_DIR, CATEGORY)
train_ds = build_dataset(DATA_DIR, CATEGORY, "train", url_mapping)
val_ds = build_dataset(DATA_DIR, CATEGORY, "val", url_mapping)

## Step 2: Baseline — BLIP-2 Without Fine-Tuning

How well does pretrained BLIP-2 handle "find this dress but make it red" queries out of the box?

In [ ]:
baseline_eval = khoji.ComposedEvaluator(MODEL)
baseline = baseline_eval.evaluate(
    dataset_name=f"fashioniq-{CATEGORY}",
    dataset=val_ds,
    k_values=[1, 5, 10, 50],
    n_queries=MAX_EVAL_QUERIES,
    cache_dir=CACHE_DIR,
)
baseline.print()
del baseline_eval

## Step 3: Fine-Tune BLIP-2 with LoRA (Standard Pipeline)

The standard approach: apply LoRA to BLIP-2's attention layers and fine-tune on composed triplets. The default fusion is simple addition: `embedding = image_emb + text_emb`.

In [ ]:
from functools import partial

# Build training triplets (random negatives for speed)
triplets = khoji.build_random_negatives_composed(train_ds, n_negatives=3, seed=42)

# Train with LoRA
lora_config = khoji.ComposedTrainingConfig(
    epochs=5,
    batch_size=8,
    grad_accum_steps=1,
    lr=2e-5,
    warmup_steps=50,
    loss_fn=partial(khoji.infonce_loss, temperature=0.05),
    lora=khoji.LoRASettings(r=8, alpha=16, dropout=0.1),
    save_dir="./output/fashioniq-lora/adapter",
    sanity_check_samples=5,
    cache_dir=CACHE_DIR,
)

lora_trainer = khoji.ComposedTrainer(MODEL, lora_config)
lora_history = lora_trainer.train(khoji.ComposedTripletDataset(triplets))
print(f"Final epoch loss: {lora_history.epoch_loss[-1]:.4f}")

In [ ]:
# Evaluate LoRA fine-tuned model
lora_eval = khoji.ComposedEvaluator(MODEL, adapter_path="./output/fashioniq-lora/adapter")
lora_result = lora_eval.evaluate(
    dataset_name=f"fashioniq-{CATEGORY}",
    dataset=val_ds,
    k_values=[1, 5, 10, 50],
    n_queries=MAX_EVAL_QUERIES,
    cache_dir=CACHE_DIR,
)
lora_result.print()
del lora_eval

## Step 4: Custom FFN Fusion Model

The default BLIP-2 composed embedding is `image_emb + text_emb` — simple addition. But addition treats both modalities equally and doesn't learn *how* to combine them.

Here we build a **custom fusion model**: we still use BLIP-2 to extract image and text embeddings, but instead of adding them, we concatenate them and pass through a learned FFN:

```
[image_emb (256) ; text_emb (256)] → Linear(512, 256) → ReLU → Linear(256, 256) → fused_emb (256)
```

This lets the model learn which modality to emphasize and how to combine the signals. We train the FFN from scratch (and optionally LoRA on the BLIP-2 backbone).

To plug this into khoji, we use `ComposedTrainer` with a custom `encode_fn` that handles image-only, text-only, and joint modes.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoProcessor, Blip2ForImageTextRetrieval
from khoji.device import get_device

device = get_device()

class FFNFusionModel(nn.Module):
    """BLIP-2 backbone + learned FFN for fusing image and text embeddings.

    Instead of: embedding = image_emb + text_emb  (default)
    We do:      embedding = FFN(concat(image_emb, text_emb))
    """

    def __init__(self, model_name, embed_dim=256):
        super().__init__()
        self.blip2 = Blip2ForImageTextRetrieval.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name)

        # Freeze BLIP-2 backbone — only train the FFN
        for param in self.blip2.parameters():
            param.requires_grad = False

        # Learned fusion: concat(img_emb, txt_emb) -> fused_emb
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def _extract_embeddings(self, images, texts):
        """Run BLIP-2 forward pass and return raw image + text embeddings."""
        inputs = self.processor(
            images=images, text=texts,
            return_tensors="pt", padding=True, truncation=True,
        ).to(next(self.parameters()).device)

        out = self.blip2(**inputs, use_image_text_matching_head=False)
        img_emb = out.image_embeds.max(dim=1).values  # [batch, 256]
        txt_emb = out.text_embeds                       # [batch, 256]
        return img_emb, txt_emb

    def encode(self, images=None, texts=None):
        """Unified encode function for ComposedTrainer.

        Handles image-only, text-only, and joint (image+text) modes.
        """
        if images is not None and texts is not None:
            # Joint mode: use FFN fusion
            img_emb, txt_emb = self._extract_embeddings(images, texts)
            return self.fusion(torch.cat([img_emb, txt_emb], dim=1))
        elif images is not None:
            # Image-only mode: gallery encoding
            dummy_texts = [""] * len(images)
            img_emb, _ = self._extract_embeddings(images, dummy_texts)
            return img_emb
        elif texts is not None:
            # Text-only mode
            from PIL import Image as PILImage
            dummy_imgs = [PILImage.new("RGB", (224, 224), "black")] * len(texts)
            _, txt_emb = self._extract_embeddings(dummy_imgs, texts)
            return txt_emb
        else:
            raise ValueError("At least one of images or texts must be provided.")


# Build the model
ffn_model = FFNFusionModel(MODEL).to(device)

# Check trainable params — only FFN should be trainable
trainable = sum(p.numel() for p in ffn_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in ffn_model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")
print(f"FFN parameters: {trainable:,}")

### Train the Custom FFN Model

We pass our `FFNFusionModel` to `ComposedTrainer` along with a custom encode function. khoji handles the training loop, optimizer, scheduler, and gradient clipping — we just define how inputs become embeddings.

The `encode_fn` receives `list[PIL.Image] | None` and `list[str] | None`. The trainer loads images from file paths before calling this function, and dispatches based on which modalities are present in each batch.

In [ ]:
# Train the FFN fusion model using ComposedTrainer with a custom encode function
ffn_config = khoji.ComposedTrainingConfig(
    epochs=5,
    batch_size=8,
    grad_accum_steps=1,
    lr=1e-3,           # higher LR since we're training a small FFN from scratch
    warmup_steps=50,
    loss_fn=partial(khoji.infonce_loss, temperature=0.05),
    lora=None,          # no LoRA — we're training the FFN directly
    save_dir="./output/fashioniq-ffn/model",
    sanity_check_samples=5,
    cache_dir=CACHE_DIR,
)

ffn_trainer = khoji.ComposedTrainer(
    model=ffn_model,                 # nn.Module with all parameters
    encode_fn=ffn_model.encode,      # (list[PIL]|None, list[str]|None) -> Tensor
    config=ffn_config,
)

ffn_history = ffn_trainer.train(khoji.ComposedTripletDataset(triplets))
print(f"Final epoch loss: {ffn_history.epoch_loss[-1]:.4f}")

### Evaluate the FFN Model

To evaluate the custom model, we wrap it in a `JointEmbeddingModel` with a custom encoder function. This encoder must handle three calling patterns: image-only, text-only, and joint.

In [ ]:
import torch.nn.functional as F

# Wrap the FFN model as a JointEmbeddingModel encoder for evaluation.
# The encoder reuses ffn_model.encode() which already handles all three modes.
ffn_model.eval()

@torch.no_grad()
def ffn_encoder(images, texts, device):
    """Custom encoder for JointEmbeddingModel — delegates to FFNFusionModel.encode()."""
    emb = ffn_model.encode(images=images, texts=texts)
    return F.normalize(emb, p=2, dim=1)

ffn_eval_model = khoji.JointEmbeddingModel(encoder=ffn_encoder)
ffn_evaluator = khoji.ComposedEvaluator(embedding_model=ffn_eval_model)

ffn_result = ffn_evaluator.evaluate(
    dataset_name=f"fashioniq-{CATEGORY}",
    dataset=val_ds,
    k_values=[1, 5, 10, 50],
    n_queries=MAX_EVAL_QUERIES,
    cache_dir=CACHE_DIR,
)
ffn_result.print()

## Step 5: Compare All Results

In [ ]:
results = {
    "BLIP-2 baseline": baseline.metrics,
    "BLIP-2 + LoRA": lora_result.metrics,
    "BLIP-2 + FFN fusion": ffn_result.metrics,
}

print(f"{'Metric':<12}" + "".join(f"{name:>20}" for name in results))
print("=" * 72)
for metric in ["recall@1", "recall@10", "recall@50", "mrr@10"]:
    row = f"{metric:<12}"
    for name, metrics in results.items():
        row += f"{metrics.get(metric, 0):>20.4f}"
    print(row)

# Highlight best approach
print("\nDelta vs baseline:")
for name in ["BLIP-2 + LoRA", "BLIP-2 + FFN fusion"]:
    deltas = []
    for metric in ["recall@10", "mrr@10"]:
        d = results[name][metric] - results["BLIP-2 baseline"][metric]
        deltas.append(f"{metric}: {d:+.4f}")
    print(f"  {name}: {', '.join(deltas)}")

## Step 6: Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

histories = {"LoRA (addition)": lora_history, "FFN fusion": ffn_history}
for name, hist in histories.items():
    axes[0].plot(hist.step_loss, label=name, alpha=0.7)
    axes[1].plot(hist.epoch_loss, marker="o", label=name)

axes[0].set_title("Loss per Step")
axes[0].set_xlabel("Optimizer Step")
axes[0].legend()

axes[1].set_title("Average Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("Composed Retrieval: LoRA vs FFN Fusion", fontsize=13)
plt.tight_layout()
plt.savefig("fashioniq_training_curves.png", dpi=150)
plt.show()

## Step 7: Inference — Query with a Reference Image + Modification

Use the fine-tuned model to find target images given a reference and modification text.

In [ ]:
import torch

# Load the LoRA fine-tuned model for inference
model = khoji.JointEmbeddingModel(MODEL, adapter_path="./output/fashioniq-lora/adapter")

# Pick a sample query from the validation set
sample_qid = list(val_ds.queries.keys())[0]
ref_image_src, mod_text = val_ds.queries[sample_qid]
target_id = list(val_ds.qrels[sample_qid].keys())[0]
target_img_src, target_txt = val_ds.corpus[target_id]

print(f"Reference image: {ref_image_src[:60]}...")
print(f"Modification: '{mod_text}'")
print(f"Target: {target_img_src[:60]}...")

# Encode the query
ref_img = khoji.load_image(ref_image_src, cache_dir=CACHE_DIR)
query_emb = model.encode(images=[ref_img], texts=[mod_text])

# Encode a small gallery
gallery_ids = list(val_ds.corpus.keys())[:200]
gallery_imgs = []
valid_gids = []
for gid in gallery_ids:
    img_src, _ = val_ds.corpus[gid]
    img = khoji.load_image(img_src, cache_dir=CACHE_DIR)
    if img is not None:
        gallery_imgs.append(img)
        valid_gids.append(gid)

gallery_emb = model.encode(images=gallery_imgs)
scores = torch.mm(query_emb, gallery_emb.t()).squeeze(0)
top5 = torch.topk(scores, k=5).indices.tolist()

print(f"\nTop 5 results:")
for rank, idx in enumerate(top5):
    gid = valid_gids[idx]
    gid_img_src, _ = val_ds.corpus[gid]
    is_target = " *** TARGET ***" if gid == target_id else ""
    print(f"  Rank {rank+1}: {gid_img_src[:60]}... (score={scores[idx]:.4f}){is_target}")

## Summary

| Model | Fusion method | Trainable params | Recall@10 | MRR@10 |
|-------|--------------|-----------------|-----------|--------|
| BLIP-2 baseline | addition | 0 | (low) | (low) |
| BLIP-2 + LoRA | addition | ~0.1% (LoRA adapters) | (improved) | (improved) |
| BLIP-2 + FFN | learned concat→FFN | ~131K (FFN only) | (compare) | (compare) |

**Key takeaways:**
- Pretrained BLIP-2 struggles with composed retrieval — it needs fine-tuning to learn "find this but different."
- **LoRA fine-tuning** with default addition fusion is the simplest approach and works well.
- **Custom FFN fusion** replaces the additive combination with a learned layer. This shows how khoji supports fully custom model architectures — you bring your own `nn.Module` and encode function, khoji handles the training loop.
- The FFN has very few parameters (~131K) but can learn richer fusion patterns than simple addition. Whether this helps depends on the dataset and how much the two modalities need to interact.

### What this notebook demonstrated
1. Building a `ComposedRetrievalDataset` from external data (FashionIQ)
2. Standard LoRA fine-tuning via `ComposedTrainer` with a HuggingFace model
3. **Custom model architecture** — loading BLIP-2 manually, adding an FFN fusion layer, and plugging it into khoji's training and evaluation pipeline via a custom `encode_fn`
4. Wrapping a custom model in `JointEmbeddingModel(encoder=...)` for evaluation